## Getting the data

Request all issues with personified token from github_apikey.key file. One can generate a token under https://github.com/settings/tokens

Downloaded data available on https://gigamove.rwth-aachen.de/de/download/5726f0a388b74818ddd8fdec187bd267

In [ ]:
import requests
import time
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
import re
import ast
import random
import statistics
import math
from pathlib import Path
from typing import List, Dict, Optional
from collections import Counter, defaultdict

In [ ]:
TOKEN = "YOUR_PERSONAL_ACCESS_TOKEN" 
if os.path.exists("github_apikey.key"):
    with open("github_apikey.key", 'r') as f:
        TOKEN = f.read().replace("\n", "")

query_template = """
query($cursor: String) {
  rateLimit {
    cost
    remaining
    resetAt
  }
  repository(owner: "%s", name: "%s") {
    issues(
      states: CLOSED, 
      first: 30, 
      after: $cursor
    ) {
      pageInfo {
        hasNextPage
        endCursor
      }
      nodes {
        number
        title
        body
        createdAt
        author {
          login
        }
        labels(first: 20) {
          nodes {
            name
          }
        }
        comments(first: 100) {
          nodes {
            author {
              login
            }
            body
          }
        }
      }
    }
  }
}
"""


In [ ]:
def run_query(owner, repo, cursor):
    url = 'https://api.github.com/graphql'
    headers = {"Authorization": f"Bearer {TOKEN}"}
    variables = {"cursor": cursor}
    
    try:
        response = requests.post(url, json={'query': query_template % (owner, repo), 'variables': variables}, headers=headers, timeout=30)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"Connection Error: {e}")
        return None

In [ ]:
def request_all_issues(owner, repo):
    has_next = True
    cursor = None
    total_issues_processed = 0

    raw_data = open("raw_data_"+repo+".txt", 'w', encoding='utf-8')

    while has_next:
        result = run_query(owner, repo, cursor)
        
        # Check for empty result or connection failure
        if not result:
            print("Network error. Sleeping 5s...")
            time.sleep(5)
            continue

        # Check for GraphQL specific errors (e.g., Bad syntax)
        if 'errors' in result:
            print(f"GraphQL Error: {result['errors']}")
            break # Break loop, otherwise it retries infinitely

        # Check if data exists
        if 'data' not in result or not result['data']:
            print("No data returned.")
            break
        
        data = result['data']
        
        if 'rateLimit' in data:
            rate_limit = data['rateLimit']
            remaining = rate_limit['remaining']
            print(f"Points remaining: {remaining} (Cost: {rate_limit['cost']})")
            
            if remaining < 100:
                print("RATE LIMIT NEAR! Sleeping for 1 hour...")
                time.sleep(3600)
        
        data_repo = data['repository']['issues']
        
        for issue in data_repo['nodes']:
            if not issue or not issue['author'] or not issue['title'] or not issue['body']: 
                continue
            
            issue_id = str(issue['number'])
            issue_author = issue['author']['login']

            issue_title = issue['title'].replace("\n", " ").replace("\r", "")
            issue_body = issue['body'].replace("\n", " ").replace("\r", "")
            issue_date = issue['createdAt']

            label_list = []
            if 'labels' in issue and issue['labels']['nodes']:
                label_list = [l['name'] for l in issue['labels']['nodes']]
            
            label_str = "|".join(label_list)

            raw_data.write(f"issue:{issue_author}\n{label_str}\n{issue_date}\n{issue_title}\n{issue_body}\n")

            if 'comments' in issue and issue['comments']['nodes']:
                for comment in issue['comments']['nodes']:
                    if not comment or not comment['author'] or not comment['body']: 
                        continue
                    
                    comment_author = comment['author']['login']
                    
                    comment_body = comment['body'].replace("\n", " ").replace("\r", "")

                    raw_data.write(f"comment:{comment_author}\n{comment_body}\n")

        # Update Progress
        total_issues_processed += len(data_repo['nodes'])
        has_next = data_repo['pageInfo']['hasNextPage']
        cursor = data_repo['pageInfo']['endCursor']
        
        print(f"Total issues saved: {total_issues_processed}")

    raw_data.close()
    print("Done")

In [ ]:
#request_all_issues("tensorflow", "tensorflow")

In [ ]:
#request_all_issues("networkx", "networkx")

In [ ]:
#request_all_issues("pandas-dev", "pandas")

## Building the graph

In [ ]:
read_raw_data = open("raw_data_tensorflow.txt", 'r', encoding='utf-8')

G_tensorflow = nx.DiGraph()

current_issue_opener = ""

for line in read_raw_data:
    if line.startswith("issue:"):
        current_issue_opener = line.strip().split(':')[1]
    elif line.startswith("comment:"):
        commentor = line.strip().split(':')[1]
        if commentor != current_issue_opener: 
            G_tensorflow.add_edge(commentor, current_issue_opener)

read_raw_data.close()
print(G_tensorflow)

In [ ]:
read_raw_data = open("raw_data_pandas.txt", 'r', encoding='utf-8')

G_pandas = nx.DiGraph()

current_issue_opener = ""

for line in read_raw_data:
    if line.startswith("issue:"):
        current_issue_opener = line.strip().split(':')[1]
    elif line.startswith("comment:"):
        commentor = line.strip().split(':')[1]
        if commentor != current_issue_opener: 
            G_pandas.add_edge(commentor, current_issue_opener)

read_raw_data.close()
print(G_pandas)

In [ ]:
read_raw_data = open("raw_data_networkx.txt", 'r', encoding='utf-8')

G_networkx = nx.DiGraph()

current_issue_opener = ""

for line in read_raw_data:
    if line.startswith("issue:"):
        current_issue_opener = line.strip().split(':')[1]
    elif line.startswith("comment:"):
        commentor = line.strip().split(':')[1]
        if commentor != current_issue_opener: 
            G_networkx.add_edge(commentor, current_issue_opener)

read_raw_data.close()
print(G_networkx)

In [ ]:
largest = max(nx.weakly_connected_components(G_tensorflow), key=len)
G_tensorflow = G_tensorflow.subgraph(largest)

largest = max(nx.weakly_connected_components(G_pandas), key=len)
G_pandas = G_pandas.subgraph(largest)

largest = max(nx.weakly_connected_components(G_networkx), key=len)
G_networkx = G_networkx.subgraph(largest)

print("G is a " + str(G_tensorflow))
print("G is a " + str(G_pandas))
print("G is a " + str(G_networkx))

In [ ]:
both_tensorflow_pandas = [d for d in G_tensorflow.nodes() if d in G_pandas]
both_tensorflow_networkx = [d for d in G_tensorflow.nodes() if d in G_networkx]
both_pandas_networkx = [d for d in G_pandas.nodes() if d in G_networkx]

print(len(both_tensorflow_pandas), len(both_tensorflow_networkx), len(both_pandas_networkx))

In [ ]:
in_degree_sequence_tensorflow = sorted([d for d in G_tensorflow.in_degree()], key=lambda x: x[1], reverse=True)
out_degree_sequence_tensorflow = sorted([d for d in G_tensorflow.out_degree()], key=lambda x: x[1], reverse=True)

in_degree_sequence_pandas = sorted([d for d in G_pandas.in_degree()], key=lambda x: x[1], reverse=True)
out_degree_sequence_pandas = sorted([d for d in G_pandas.out_degree()], key=lambda x: x[1], reverse=True)

in_degree_sequence_networkx = sorted([d for d in G_networkx.in_degree()], key=lambda x: x[1], reverse=True)
out_degree_sequence_networkx = sorted([d for d in G_networkx.out_degree()], key=lambda x: x[1], reverse=True)

df_top10_in_tensorflow = pd.DataFrame(in_degree_sequence_tensorflow[0:10], columns=('User', ''))
df_top10_out_tensorflow = pd.DataFrame(out_degree_sequence_tensorflow[0:10], columns=('User', ''))

df_top10_in_pandas = pd.DataFrame(in_degree_sequence_pandas[0:10], columns=('User', ''))
df_top10_out_pandas = pd.DataFrame(out_degree_sequence_pandas[0:10], columns=('User', ''))

df_top10_in_networkx = pd.DataFrame(in_degree_sequence_networkx[0:10], columns=('User', ''))
df_top10_out_networkx = pd.DataFrame(out_degree_sequence_networkx[0:10], columns=('User', ''))

frames = {
    ('TensorFlow', 'In-Degree'): df_top10_in_tensorflow,
    ('TensorFlow', 'Out-Degree'): df_top10_out_tensorflow,
    ('Pandas', 'In-Degree'): df_top10_in_pandas,
    ('Pandas', 'Out-Degree'): df_top10_out_pandas,
    ('NetworkX', 'In-Degree'): df_top10_in_networkx,
    ('NetworkX', 'Out-Degree'): df_top10_out_networkx,
}

combined_df = pd.concat(
    [df.reset_index(drop=True) for df in frames.values()], 
    axis=1, 
    keys=frames.keys()
)

print(combined_df.to_string(index=False))

In [ ]:
print(f"The median of the in degree is {in_degree_sequence_tensorflow[int(len(in_degree_sequence_tensorflow)/2)][1]} and also for the out degree it is {out_degree_sequence_tensorflow[int(len(out_degree_sequence_tensorflow)/2)][1]}.")
print(f"The average degree is {sum([x[1] for x in in_degree_sequence_tensorflow])/len(in_degree_sequence_tensorflow):.4f}.")

In [ ]:
print(f"The median of the in degree is {in_degree_sequence_pandas[int(len(in_degree_sequence_pandas)/2)][1]} and also for the out degree it is {out_degree_sequence_pandas[int(len(out_degree_sequence_pandas)/2)][1]}.")
print(f"The average degree is {sum([x[1] for x in in_degree_sequence_pandas])/len(in_degree_sequence_pandas):.4f}.")

In [ ]:
print(f"The median of the in degree is {in_degree_sequence_networkx[int(len(in_degree_sequence_networkx)/2)][1]} and also for the out degree it is {out_degree_sequence_networkx[int(len(out_degree_sequence_networkx)/2)][1]}.")
print(f"The average degree is {sum([x[1] for x in in_degree_sequence_networkx])/len(in_degree_sequence_networkx):.4f}.")

In [ ]:
number_bins = 40

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Tensorflow In-degree
axes[0, 0].hist([x[1] for x in in_degree_sequence_tensorflow], bins=number_bins)
axes[0, 0].set_title("In-degree: Tensorflow")
axes[0, 0].set_xlabel("In-degree")
axes[0, 0].set_ylabel("Frequency")

# Pandas In-degree
axes[0, 1].hist([x[1] for x in in_degree_sequence_pandas], bins=number_bins)
axes[0, 1].set_title("In-degree: Pandas")
axes[0, 1].set_xlabel("In-degree")

# NetworkX In-degree
axes[0, 2].hist([x[1] for x in in_degree_sequence_networkx], bins=number_bins)
axes[0, 2].set_title("In-degree: NetworkX")
axes[0, 2].set_xlabel("In-degree")


# Tensorflow Out-degree
axes[1, 0].hist([x[1] for x in out_degree_sequence_tensorflow], bins=number_bins)
axes[1, 0].set_title("Out-degree: Tensorflow")
axes[1, 0].set_xlabel("Out-degree")
axes[1, 0].set_ylabel("Frequency")

# Pandas Out-degree
axes[1, 1].hist([x[1] for x in out_degree_sequence_pandas], bins=number_bins)
axes[1, 1].set_title("Out-degree: Pandas")
axes[1, 1].set_xlabel("Out-degree")

# NetworkX Out-degree
axes[1, 2].hist([x[1] for x in out_degree_sequence_networkx], bins=number_bins)
axes[1, 2].set_title("Out-degree: NetworkX")
axes[1, 2].set_xlabel("Out-degree")

plt.tight_layout()
plt.show()

In [ ]:
print(f"The number of nodes that have an in degree of 1 is {len([x for x in in_degree_sequence_tensorflow if x[1]<=1])/len(in_degree_sequence_tensorflow)*100:.2f}%.")
print(f"The number of nodes that have an out degree of 1 is {len([x for x in out_degree_sequence_tensorflow if x[1]<=1])/len(out_degree_sequence_tensorflow)*100:.2f}%.")

In [ ]:
print(f"The number of nodes that have an in degree of 1 is {len([x for x in in_degree_sequence_pandas if x[1]<=1])/len(in_degree_sequence_pandas)*100:.2f}%.")
print(f"The number of nodes that have an out degree of 1 is {len([x for x in out_degree_sequence_pandas if x[1]<=1])/len(out_degree_sequence_pandas)*100:.2f}%.")

In [ ]:
print(f"The number of nodes that have an in degree of 1 is {len([x for x in in_degree_sequence_networkx if x[1]<=1])/len(in_degree_sequence_networkx)*100:.2f}%.")
print(f"The number of nodes that have an out degree of 1 is {len([x for x in out_degree_sequence_networkx if x[1]<=1])/len(out_degree_sequence_networkx)*100:.2f}%.")

In [ ]:
cutoff = 10

print(f"The number of nodes that have an in degree of less than {cutoff} is {len([x for x in in_degree_sequence_tensorflow if x[1]<cutoff])/len(in_degree_sequence_tensorflow)*100:.2f}%.")
print(f"The number of nodes that have an out degree of less than {cutoff} is {len([x for x in out_degree_sequence_tensorflow if x[1]<cutoff])/len(out_degree_sequence_tensorflow)*100:.2f}%.")

In [ ]:
print(f"The number of nodes that have an in degree of less than {cutoff} is {len([x for x in in_degree_sequence_pandas if x[1]<cutoff])/len(in_degree_sequence_pandas)*100:.2f}%.")
print(f"The number of nodes that have an out degree of less than {cutoff} is {len([x for x in out_degree_sequence_pandas if x[1]<cutoff])/len(out_degree_sequence_pandas)*100:.2f}%.")

In [ ]:
print(f"The number of nodes that have an in degree of less than {cutoff} is {len([x for x in in_degree_sequence_networkx if x[1]<cutoff])/len(in_degree_sequence_networkx)*100:.2f}%.")
print(f"The number of nodes that have an out degree of less than {cutoff} is {len([x for x in out_degree_sequence_networkx if x[1]<cutoff])/len(out_degree_sequence_networkx)*100:.2f}%.")

In [ ]:
in_tensorflow_greater_cutoff = [d for d in G_tensorflow.in_degree() if d[1] >= cutoff]
out_tensorflow_greater_cutoff = [d for d in G_tensorflow.out_degree() if d[1] >= cutoff]

in_pandas_greater_cutoff = [d for d in G_pandas.in_degree() if d[1] >= cutoff]
out_pandas_greater_cutoff = [d for d in G_pandas.out_degree() if d[1] >= cutoff]

in_networkx_greater_cutoff = [d for d in G_networkx.in_degree() if d[1] >= cutoff]
out_networkx_greater_cutoff = [d for d in G_networkx.out_degree() if d[1] >= cutoff]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Tensorflow In-degree
axes[0, 0].hist([x[1] for x in in_tensorflow_greater_cutoff], bins=number_bins)
axes[0, 0].set_title("In-degree after cutoff: Tensorflow")
axes[0, 0].set_xlabel("In-degree")
axes[0, 0].set_ylabel("Frequency")

# Pandas In-degree
axes[0, 1].hist([x[1] for x in in_pandas_greater_cutoff], bins=number_bins)
axes[0, 1].set_title("In-degree after cutoff: Pandas")
axes[0, 1].set_xlabel("In-degree")

# NetworkX In-degree
axes[0, 2].hist([x[1] for x in in_networkx_greater_cutoff], bins=number_bins)
axes[0, 2].set_title("In-degree after cutoff: NetworkX")
axes[0, 2].set_xlabel("In-degree")


# Tensorflow Out-degree
axes[1, 0].hist([x[1] for x in out_tensorflow_greater_cutoff], bins=number_bins)
axes[1, 0].set_title("Out-degree after cutoff: Tensorflow")
axes[1, 0].set_xlabel("Out-degree")
axes[1, 0].set_ylabel("Frequency")

# Pandas Out-degree
axes[1, 1].hist([x[1] for x in out_pandas_greater_cutoff], bins=number_bins)
axes[1, 1].set_title("Out-degree after cutoff: Pandas")
axes[1, 1].set_xlabel("Out-degree")

# NetworkX Out-degree
axes[1, 2].hist([x[1] for x in out_networkx_greater_cutoff], bins=number_bins)
axes[1, 2].set_title("Out-degree after cutoff: NetworkX")
axes[1, 2].set_xlabel("Out-degree")

plt.tight_layout()
plt.show()

In [ ]:
G_mod = nx.DiGraph(G_networkx)

for (user, number_nodes) in in_degree_sequence:
    if number_nodes == 0:
        G_mod.remove_node(user)

print(G)

In [ ]:
print(G_mod)

In [ ]:
for (user, number_nodes) in out_degree_sequence:
    if number_nodes == 0:
        G_mod.remove_node(user)

print(G)
print(G_mod)

In [ ]:
largest = max(nx.weakly_connected_components(G_mod), key=len)

G_mod = G_mod.subgraph(largest)

print("G is a " + str(G_mod))

In [ ]:
in_degree_sequence = sorted([d for d in G_mod.in_degree()], key = (lambda x: x[1]))
out_degree_sequence = sorted([d for d in G_mod.out_degree()], key = (lambda x: x[1]))

number_bins = 35

plt.hist([x[1] for x in in_degree_sequence], bins=number_bins)
plt.xlabel("In-degree")
plt.ylabel("Frequency")
plt.title("In-degree frequency of the issues network")
plt.show()

plt.hist([x[1] for x in out_degree_sequence], bins=number_bins)
plt.xlabel("Out-degree")
plt.ylabel("Frequency")
plt.title("Out-degree frequency of the issues network")
plt.show()

In [ ]:
G_und = G_mod.to_undirected()

pose = nx.forceatlas2_layout(G_und)

In [ ]:
plt.figure(figsize=(15, 15))

nx.draw(
    G_und,
    pos=pose,
    node_size=10,
    alpha=0.8,
    edge_color='gray',
    width=0.5,
    with_labels=False
)

plt.title("First visualization of the network with ForceAtlas2")
plt.axis('off')
plt.show()

In [ ]:
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

nodes = list(G.nodes())
x_out = [out_degrees[n] for n in nodes if out_degrees[n] < 1500 and in_degrees[n] < 200 and out_degrees[n] > 20 and in_degrees[n] > 10]
y_in  = [in_degrees[n] for n in nodes if in_degrees[n] < 200 and out_degrees[n] < 1500 and out_degrees[n] > 20 and in_degrees[n] > 10]

print(len(x_out))

plt.figure(figsize=(10, 8))
plt.scatter(x_out, y_in, alpha=0.6, s=30)

plt.xlabel("Out-degree")
plt.ylabel("In-degree")
plt.title("Scatter plot of In-degree vs Out-degree")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
G_weighted = nx.DiGraph()

read_raw_data = open("raw_data.txt", 'r', encoding='utf-8')

for line in read_raw_data:
    line = line.strip()
    if not line: continue # Skip empty lines

    if line.startswith("issue:"):
        current_issue_opener = line.split(':')[1]
    elif line.startswith("comment:"):
        commentor = line.split(':')[1]
        
        # Prevent self-loops (commenting on own issue)
        if commentor != current_issue_opener and current_issue_opener != "": 
            
            # Logic to handle weights
            if G_weighted.has_edge(commentor, current_issue_opener):
                # If edge exists, increment weight
                G_weighted[commentor][current_issue_opener]['weight'] += 1
            else:
                # If edge doesn't exist, create it with weight 1
                G_weighted.add_edge(commentor, current_issue_opener, weight=1)

In [ ]:
in_degree_sequence = sorted([d for d in G_weighted.in_degree()], key = (lambda x: x[1]), reverse=True)

out_degree_sequence = sorted([d for d in G_weighted.out_degree()], key = (lambda x: x[1]), reverse=True)

print(in_degree_sequence[0:50])
print(out_degree_sequence[0:50])

In [ ]:
bots = ['google-ml-butler', 'tensorflowbutler', 'github-actions', 'tensorflow-bot']
G_weighted.remove_nodes_from(bots)

In [ ]:
# Code from https://www.michelecoscia.com/?page_id=287
def disparity_filter(table, undirected = False, return_self_loops = False):
   # sys.stderr.write("Calculating DF score...\n")
   table = table.copy()
   table_sum = table.groupby(table["src"]).sum().reset_index()
   table_deg = table.groupby(table["src"]).count()["trg"].reset_index()
   table = table.merge(table_sum, on = "src", how = "left", suffixes = ("", "_sum"))
   table = table.merge(table_deg, on = "src", how = "left", suffixes = ("", "_count"))
   table["score"] = 1.0 - ((1.0 - (table["nij"] / table["nij_sum"])) ** (table["trg_count"] - 1))
   table["variance"] = (table["trg_count"] ** 2) * (((20 + (4.0 * table["trg_count"])) / ((table["trg_count"] + 1.0) * (table["trg_count"] + 2) * (table["trg_count"] + 3))) - ((4.0) / ((table["trg_count"] + 1.0) ** 2)))
   if not return_self_loops:
      table = table[table["src"] != table["trg"]]
   if undirected:
      table["edge"] = table.apply(lambda x: "%s-%s" % (min(x["src"], x["trg"]), max(x["src"], x["trg"])), axis = 1)
      table_maxscore = table.groupby(by = "edge")["score"].max().reset_index()
      table_minvar = table.groupby(by = "edge")["variance"].min().reset_index()
      table = table.merge(table_maxscore, on = "edge", suffixes = ("_min", ""))
      table = table.merge(table_minvar, on = "edge", suffixes = ("_max", ""))
      table = table.drop_duplicates(subset = ["edge"])
      
      # *** CORRECTION APPLIED HERE ***
      table = table.drop("edge", axis=1)
      table = table.drop("score_min", axis=1)
      table = table.drop("variance_max", axis=1)
      # *******************************
      
   return table[["src", "trg", "nij", "score", "variance"]]

def high_salience_skeleton(table, undirected = False, return_self_loops = False):
   sys.stderr.write("Calculating HSS score...\n")
   table = table.copy()
   table["distance"] = 1.0 / table["nij"]
   nodes = set(table["src"]) | set(table["trg"])
   G = nx.from_pandas_edgelist(table, source = "src", target = "trg", edge_attr = "distance", create_using = nx.DiGraph())
   cs = defaultdict(float)
   for s in nodes:
      pred = defaultdict(list)
      dist = {t: float("inf") for t in nodes}
      dist[s] = 0.0
      Q = defaultdict(list)
      for w in dist:
         Q[dist[w]].append(w)
      S = []
      while len(Q) > 0:
         v = Q[min(Q.keys())].pop(0)
         S.append(v)
         for _, w, l in G.edges(nbunch = [v,], data = True):
            new_distance = dist[v] + l["distance"]
            if dist[w] > new_distance:
               Q[dist[w]].remove(w)
               dist[w] = new_distance
               Q[dist[w]].append(w)
               pred[w] = []
            if dist[w] == new_distance:
               pred[w].append(v)
         while len(S) > 0:
            w = S.pop()
            for v in pred[w]:
               cs[(v, w)] += 1.0
         Q = defaultdict(list, {k: v for k, v in Q.items() if len(v) > 0})
   table["score"] = table.apply(lambda x: cs[(x["src"], x["trg"])] / len(nodes), axis = 1)
   if not return_self_loops:
      table = table[table["src"] != table["trg"]]
   if undirected:
      table["edge"] = table.apply(lambda x: "%s-%s" % (min(x["src"], x["trg"]), max(x["src"], x["trg"])), axis = 1)
      table_maxscore = table.groupby(by = "edge")["score"].sum().reset_index()
      table = table.merge(table_maxscore, on = "edge", suffixes = ("_min", ""))
      table = table.drop_duplicates(subset = ["edge"])
      table = table.drop("edge", 1)
      table = table.drop("score_min", 1)
      table["score"] = table["score"] / 2.0
   return table[["src", "trg", "nij", "score"]]

In [ ]:
edge_data = []
for u, v, data in G_weighted.edges(data=True):
    edge_data.append({
        "src": u, 
        "trg": v, 
        "nij": data['weight']
    })

df_edges = pd.DataFrame(edge_data)

if not df_edges.empty:
    # 2. Run the disparity filter
    # The function returns the table with a 'score' column (p-value equivalent)
    df_scored = disparity_filter(df_edges, undirected=False)
    
    # 3. Filter Edges
    # We keep edges where the score is below a significance level (alpha).
    # Typical values are 0.05 or 0.5 depending on how aggressive you want to be.
    # Since this is a small dummy dataset, we use a loose alpha (1.0) to keep everything 
    # for demo purposes, but in practice, you set this to e.g., 0.5.
    alpha = 0.3
    
    # NOTE: In disparity filter, LOW score means HIGH significance.
    # However, for nodes with degree 1, the formula often returns score 0.0 (keep).
    df_filtered = df_scored[df_scored['score'] < alpha]

    print("\n--- Disparity Filter Scores ---")
    print(df_scored[['src', 'trg', 'nij', 'score']])
    
    print(f"\nEdges kept (alpha < {alpha}): {len(df_filtered)}")

    G_filtered = nx.DiGraph()

    for index, row in df_filtered.iterrows():
        G_filtered.add_edge(row['src'], row['trg'], weight=row['nij'])


In [ ]:
print(G_filtered)
largest = max(nx.weakly_connected_components(G_filtered), key=len)

G_mod = G_filtered.subgraph(largest)

print("G is a " + str(G_mod))

In [ ]:
G_und = G_mod.to_undirected()

pose = nx.forceatlas2_layout(G_und)

In [ ]:
plt.figure(figsize=(15, 15))

nx.draw(
    G_und,
    pos=pose,
    node_size=10,
    alpha=0.8,
    edge_color='gray',
    width=0.5,
    with_labels=False
)

plt.title("First visualization of the network with ForceAtlas2")
plt.axis('off')
plt.show()

In [ ]:
print(nx.average_shortest_path_length(G_und))

In [ ]:
in_degree_sequence = sorted([d for d in G_filtered.in_degree()], key = (lambda x: x[1]), reverse=True)

out_degree_sequence = sorted([d for d in G_filtered.out_degree()], key = (lambda x: x[1]), reverse=True)

print(in_degree_sequence[0:50])
print(out_degree_sequence[0:50])